## Introduction
The study focuses on how a chromatin regulator, the histone deacetylase HDAC1, controls the development of different dendritic cell (DC) subsets and how this affects anti-tumor immune responses. Dendritic cells are crucial for activating T cells against cancer, but it was not clear how specific chromatin-modifying enzymes shape the transcription factor networks that drive DC lineage choices such as cDC1, cDC2, and pDCs. The authors therefore knockout HDAC1 and compared it with HDAC2 in mouse hematopoietic and DC compartments to see how DC development, T cell activation, and tumor control changed. To understand the mechanism, RNA-sequencing was used to measure the gene expression, ATAC-seq to assess chromatin accessibility, and CUT&RUN to map histone acetylation, allowing them to link HDAC1 loss to altered accessibility and expression at key transcription factor genes (like IRF4, IRF8, and SPIB) that determine DC subset fate and, ultimately, the strength of anti-tumor immunity. Here, the bioinformatics analysis of ATAC-sequencing is done to reproduce figure 6a-f and provide an overview of the experiment. 

## Quality Control
### Sequencing Quality Control
MultiQC v1.19 and FastQC v0.11.9 showed generally high sequencing quality across all eight ATAC‑seq libraries, with six samples passing all core FastQC modules and only two raising specific warnings. cDC1_KO_2 failed the Per base sequence content check and cDC1_WT_2 failed Adapter content, consistent with slight base‑composition bias and residual adapter contamination, while all other sections (per‑base and per‑sequence quality, GC content, N content, length distribution, and duplication) were green for every sample and GC content remained tightly clustered between 47.0% and 49.0%. Overrepresented sequence analysis confirmed that adapter‑like reads were rare: the most abundant sequence occurred 130,758 times (0.0553% of reads) in four libraries and the second sequence 28,961 times (0.0122%) in one library, with all samples having <1% of reads composed of overrepresented sequences. To address the adapter and quality issues, we trimmed low‑quality bases and adapters with Trimmomatic v0.39, which dropped 14.8–24.9% of reads (0.4–0.5% in the Trimmomatic summary post‑filtering), leaving 26–34 million surviving reads per sample with uniform 50–51 bp read length and improved quality profiles. Together, these QC metrics indicate that after trimming the libraries are technically robust, free of major contamination, and well suited for downstream alignment and chromatin accessibility analysis.

### Alignment Statistics
Samtools flagstat v1.18 and the MultiQC v1.19 general statistics table indicated very strong alignment performance to the mm10 genome for all samples. Total sequence counts ranged from 22.4 million reads in cDC2_WT_1 to 33.8 million reads in cDC1_WT_2, with mapped read counts between 16.5 million (cDC2_WT_1) and 23.8 million (cDC1_KO_1), and the fraction of unmapped reads was extremely low at 0.4–0.5% for every library, corresponding to more than 9.5% alignment efficiency. The number of uniquely mapped reads after trimming ranged from 19.6 to 28.5 million per sample, and duplication rates were moderate, spanning 14.6% in cDC1_KO_1 to 24.7% in cDC2_KO_2, with most libraries clustered around 15.5–19.3%. After filtering duplicate and mitochondrial reads, each library retained tens of millions of high‑quality nuclear fragments, providing strong signal‑to‑noise for peak calling, TSS‑based QC, and exploratory differential accessibility analyses and increasing confidence in the biological interpretability of the identified accessible chromatin regions.

### Mitochondrial Read Fraction 
Mitochondrial read fraction was quantified as an ATAC‑seq–specific QC metric. Across samples, mtDNA reads ranged from 0.5% in cDC1_KO_1 (123,567 of 23.9M total reads) to 4.7% in cDC2_KO_2 (942,657 of 20.2M reads), with most libraries falling between ~0.5% and 3%. Samples with higher mitochondrial fractions (notably cDC2_KO_2 at 4.7% and cDC2_WT_2 at 3.1%) likely reflect some degree of cell stress or compromised nuclei, as expected for ATAC‑seq, but all values remain within a range that still provides ample nuclear reads for peak calling and downstream analyses.



| Sample      | Total Reads | Motochondrial Reads | Mitochondrial Fraction |
|-------------|-------------|------------|---------------|
| cDC1_KO_1   | 23881269    | 123567     | 0.00517422    |
| cDC1_KO_2   | 23808167    | 285166     | 0.0119777     |
| cDC1_WT_1   | 22562160    | 117431     | 0.00520478    |
| cDC1_WT_2   | 23392763    | 125589     | 0.00536871    |
| cDC2_KO_1   | 22789684    | 508428     | 0.0223096     |
| cDC2_KO_2   | 20206434    | 942657     | 0.0466513     |
| cDC2_WT_1   | 16795019    | 332132     | 0.0197756     |
| cDC2_WT_2   | 18677978    | 573176     | 0.0306873     |

Table 1. Mitochondrial read fractions across ATAC-seq libraries.

### Fraction of Reads in Peaks (FRiP)
FRiP was consistently high across all libraries, with values ranging from 0.1857 in cDC1_WT_2 to 0.2476 in cDC2_KO_2 and most samples clustering between ~0.20 and 0.24. These values indicate that roughly one‑fifth to one‑quarter of all usable reads fall within consensus ATAC‑seq peaks, which is characteristic of datasets with good signal‑to‑noise rather than largely background signal. The similar FRiP scores between WT and KO samples within each cell type suggest that Hdac1 deletion does not globally degrade library quality or reduce overall enrichment in accessible chromatin, allowing downstream comparisons of accessibility to focus on true biological differences rather than technical variability.

| Sample      | in Peaks | Total Reads | FRiP  |
|-------------|----------|-------------|--------|
| cDC1_WT_1   | 4550049  | 22444729    | 0.2027 |
| cDC2_WT_2   | 3450639  | 18104802    | 0.1906 |
| cDC2_WT_1   | 3920810  | 16462887    | 0.2382 |
| cDC1_WT_2   | 4320116  | 23267174    | 0.1857 |
| cDC2_KO_2   | 4769399  | 19263777    | 0.2476 |
| cDC2_KO_1   | 5121254  | 22281256    | 0.2298 |
| cDC1_KO_1   | 4680861  | 23757702    | 0.1970 |
| cDC1_KO_2   | 5527084  | 23523001    | 0.2350 |

Table 2. Fraction of reads in peaks (FRiP) across ATAC‑seq libraries.



### Percentage of Reads within ±1 kb of Transcription Start Sites
The percentage of reads around TSSs provided another ATAC‑seq–specific QC check. For each library, approximately 7–9% of nuclear reads overlapped a ±1 kb window around annotated mm10 TSSs, indicating a strong enrichment of ATAC‑seq signal at promoter‑proximal regions rather than being uniformly distributed across the genome. These promoter‑overlap fractions were very similar between WT and KO replicates within both cDC1 and cDC2, supporting the conclusion that global promoter accessibility and library enrichment are comparable across conditions and that any Hdac1‑dependent effects are likely localized to specific regulatory loci.

<img src="./figures_tables/cDC1_KO_1_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC1_KO_2_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC1_WT_1_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC1_WT_2_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC2_KO_1_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC2_KO_2_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC2_WT_1_tss_enrichment.png" width="400" height="400">
<img src="./figures_tables/cDC2_WT_2_tss_enrichment.png" width="400" height="400">

Figure . Percentage of reads overlapping ±1 kb windows around TSSs for each ATAC‑seq library.

## Results
Differential accessibility analysis using DiffBind and edgeR identified a focused set of high-confidence DARs in both cDC1 and cDC2 at a raw P‑value threshold of 0.01, after requiring peaks to be present in at least two samples and supported by ≥10 reads in total. For each cell type, we separated significant sites into peaks that gained accessibility in KO relative to WT (logFC > 0) and peaks that lost accessibility (logFC < 0), and used these subsets for downstream motif and TSS heatmap analyses. This stringent filtering strategy yielded DARs that are strongly enriched near transcription start sites and display clear, condition-dependent changes in ATAC‑seq signal across replicates, providing a robust basis for interpreting chromatin remodeling between WT and KO dendritic cells.

<img src="./figures_tables/cDC1_gain_loss_heatmap.png">
<img src="./figures_tables/cDC2_gain_loss_heatmap.png">


### Motif Analysis
Known motif enrichment analysis of differentially accessible regions revealed a highly similar transcription factor signature in both cDC1 and cDC2 peak sets, with strong enrichment of ETS family motifs in each cell type. In cDC1, the top enriched known motifs included PU.1, SpiB, ELF5, Elf4, ETS1, and IRF/ETS-related motifs, indicating that cDC1 differential peaks are strongly associated with canonical dendritic-cell and myeloid lineage regulatory programs. Similarly, cDC2 peaks showed strong enrichment for PU.1, SpiB, ELF5, Elf4, ELF3, ETS1, and EHF motifs, again highlighting dominant ETS-family activity and suggesting that chromatin changes in cDC2 are also driven by core immune transcriptional regulators. Together, these results indicate that both cDC1 and cDC2 differentially accessible regions are enriched for motifs recognized by ETS-associated factors, with cDC1 additionally showing prominent AP-1 and BATF-related signatures among the top motifs, consistent with cell-type-specific differences superimposed on a shared dendritic-cell regulatory framework.

<img src="./figures_tables/homerknown_cdc1.png">
<img src="./figures_tables/homerknown_cdc2.png">

### Enrichment Analysis
Pathway enrichment of genes associated with differentially accessible regions revealed distinct signaling programs in cDC1 and cDC2. In cDC2, Reactome analysis highlighted strong enrichment for RHOA and RAC1 GTPase cycles, RAF–MAPK and MAPK1/MAPK3 signaling, G‑alpha (12/13) signaling, and more general “signal transduction” pathways, indicating that chromatin changes in cDC2 are preferentially located near genes controlling cytoskeletal dynamics and MAPK‑dependent signal transmission. In contrast, cDC1‑associated genes showed top enrichment for “NOTCH1 intracellular domain regulates transcription”, “Notch–HLH transcription pathway”, and “signaling by NOTCH1”, together with ERBB2/ERBB4 and type I interferon signaling modules, suggesting that cDC1 differential accessibility impacts Notch‑driven transcriptional circuits and specific cytokine/RTK signaling cascades. Together, these pathway signatures support a model in which loss of H1 modulates distinct signaling and transcriptional networks in cDC1 versus cDC2, with cDC1 changes centering on Notch and interferon pathways and cDC2 changes emphasizing Rho/MAPK‑mediated signal transduction.

<img src="./figures_tables/cdc1_reactome.png">
Figure . Reactome pathway enrichment of cDC1 differential accessibility–associated genes.

<img src="./figures_tables/cdc2_reactome.png">
Figure . Reactome pathway enrichment of cDC2 differential accessibility–associated genes.